In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import pymc as pm

from pymc_extras.marginal import marginalize

## The original model

In [ ]:
# TODO: Try to handle np.nan
# fmt: off
disaster_data = pd.Series(
    [4, 5, 4, 0, 1, 4, 3, 4, 0, 6, 3, 3, 4, 0, 2, 6,
     3, 3, 5, 4, 5, 3, 1, 4, 4, 1, 5, 5, 3, 4, 2, 5,
     2, 2, 3, 4, 2, 1, 3, np.nan, 2, 1, 1, 1, 1, 3, 0, 0,
     1, 0, 1, 1, 0, 0, 3, 1, 0, 3, 2, 2, 0, 1, 1, 1,
     0, 1, 0, 1, 0, 0, 0, 2, 1, 0, 0, 0, 1, 1, 0, 2,
     3, 3, 1, np.nan, 2, 1, 1, 1, 1, 2, 4, 2, 0, 0, 1, 4,
     0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1]
)
# fmt: on

years = np.arange(1851, 1962)

In [ ]:
with pm.Model() as disaster_model:
    switchpoint = pm.DiscreteUniform("switchpoint", lower=years.min(), upper=years.max())

    early_rate = pm.Exponential("early_rate", 1.0)
    late_rate = pm.Exponential("late_rate", 1.0)
    rate = pm.math.switch(switchpoint >= years, early_rate, late_rate)

    disasters = pm.Poisson("disasters", rate, observed=disaster_data)

In [ ]:
pm.model_to_graphviz(disaster_model)

In [ ]:
with disaster_model:
    trace = pm.sample()

In [ ]:
az.plot_dist(trace, var_names=["~switchpoint", "~disasters"]);

In [ ]:
az.summary(trace, var_names=["~switchpoint", "~disasters"])

## Marginalized model

In [ ]:
marginal_model = marginalize(disaster_model, ["switchpoint"])

In [ ]:
pm.model_to_graphviz(marginal_model)

In [ ]:
with marginal_model:
    trace = pm.sample()

In [ ]:
az.plot_dist(trace, var_names="~disasters");

In [ ]:
az.summary(trace, var_names="~disasters")